# 03 EDA

Objective: explore which patient segments and encounter patterns are most associated with 30-day readmission.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

clean_path = ROOT / 'data' / 'processed' / 'clean_diabetes.csv'
df = pd.read_csv(clean_path)
sns.set_theme(style='whitegrid')

In [ ]:
kpis = {
    'total_encounters': len(df),
    'readmission_30_rate_pct': round(df['readmitted_30'].mean() * 100, 2),
    'avg_time_in_hospital': round(df['time_in_hospital'].mean(), 2),
    'avg_num_medications': round(df['num_medications'].mean(), 2),
}
kpis

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='readmitted', order=['NO', '>30', '<30'], ax=ax)
ax.set_title('Readmission Distribution')
ax.set_xlabel('Readmission Category')
ax.set_ylabel('Count')
plt.tight_layout()

In [ ]:
age_summary = (
    df.groupby('age', as_index=False)['readmitted_30']
      .mean()
      .sort_values('age')
)
age_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=age_summary, x='age', y='readmitted_30', color='#2a6f97', ax=ax)
ax.set_title('30-Day Readmission Rate by Age Group')
ax.set_xlabel('Age Group')
ax.set_ylabel('Readmission Rate')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

In [ ]:
driver_views = {
    'admission_type': df.groupby('admission_type')['readmitted_30'].mean().sort_values(ascending=False),
    'prior_inpatient_group': df.groupby('prior_inpatient_group')['readmitted_30'].mean().sort_index(),
    'prior_emergency_group': df.groupby('prior_emergency_group')['readmitted_30'].mean().sort_index(),
    'insulin': df.groupby('insulin')['readmitted_30'].mean().sort_values(ascending=False),
}
driver_views

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plots = [
    ('admission_type', axes[0, 0]),
    ('prior_inpatient_group', axes[0, 1]),
    ('prior_emergency_group', axes[1, 0]),
    ('insulin', axes[1, 1]),
]

for key, ax in plots:
    series = driver_views[key]
    sns.barplot(x=series.index, y=series.values, color='#468faf', ax=ax)
    ax.set_title(f'Readmission Rate by {key}')
    ax.set_xlabel('')
    ax.set_ylabel('Rate')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()